In [ ]:
import pandas as pd
import numpy as np

print("开始构建多因子数据宽表...")

# ==========================================
# 第一步：读取所有数据，并统一列名为小写，方便处理
# ==========================================
print("1. 正在读取数据...")

# 增加 low_memory=False 消除 DtypeWarning
daily_stock = pd.read_csv('daily_stock.csv', low_memory=False)
daily_stock.columns = daily_stock.columns.str.lower()

# 【关键修复】：先转成字符串，并明确指定格式为 年月日 (%Y%m%d)
daily_stock['dlycaldt'] = pd.to_datetime(daily_stock['dlycaldt'].astype(str), format='%Y%m%d')

fund_q = pd.read_csv('Fundamentals Quarterly.csv')
fund_q.columns = fund_q.columns.str.lower()
fund_q['rdq'] = pd.to_datetime(fund_q['rdq']) 

link = pd.read_csv('link.csv')
link.columns = link.columns.str.lower()

macro = pd.read_csv('macro_features.csv')
macro.columns = macro.columns.str.lower()
macro['date'] = pd.to_datetime(macro['date'])

# 同样增加 low_memory=False 消除警告
sp500 = pd.read_csv('sp500_constituents.csv', low_memory=False)
sp500.columns = sp500.columns.str.lower()


# ==========================================
# 第二步：清洗 CCM 连结表，建立 GVKEY 到 PERMNO 的翻译字典
# ==========================================
print("2. 正在处理 CCM 连结表...")
# 只保留有效的链接类型 (LC, LU) 和主要链接 (P, C)
valid_link = link[
    (link['linktype'].isin(['LC', 'LU', 'lc', 'lu'])) & 
    (link['linkprim'].isin(['P', 'C', 'p', 'c']))
].copy()


# 加上 errors='coerce' 强行容错，处理异常字符
valid_link['linkdt'] = pd.to_datetime(valid_link['linkdt'], errors='coerce')

# 遇到 "E" 会变成 NaT，然后被 fillna 完美替换为今天的日期
valid_link['linkenddt'] = pd.to_datetime(valid_link['linkenddt'], errors='coerce').fillna(pd.to_datetime('today'))


# ==========================================
# 第三步：给财务数据 (GVKEY) 贴上 PERMNO 标签
# ==========================================
print("3. 正在将财务数据翻译为 PERMNO...")
# 将财务数据与连结表合并
fund_linked = pd.merge(fund_q, valid_link[['gvkey', 'lpermno', 'linkdt', 'linkenddt']], on='gvkey', how='inner')

# 核心过滤：确保财报发布日(rdq) 刚好落在这个连结(link)的有效时间段内
fund_linked = fund_linked[
    (fund_linked['rdq'] >= fund_linked['linkdt']) & 
    (fund_linked['rdq'] <= fund_linked['linkenddt'])
]

# 重命名 lpermno 为 permno，剔除多余列，并去掉没有 rdq 的残缺行
fund_linked = fund_linked.rename(columns={'lpermno': 'permno'})
fund_linked = fund_linked.dropna(subset=['rdq'])
fund_linked = fund_linked.drop(columns=['linkdt', 'linkenddt', 'gvkey', 'datadate'], errors='ignore')


# ==========================================
# 第四步：终极融合！日度量价 + 季度财务 (Point-in-Time)
# ==========================================
print("4. 正在进行 Point-in-Time 频率合并 (这步可能需要几秒到十几秒)...")
# 强制统一时间精度为纳秒 (ns)
daily_stock['dlycaldt'] = daily_stock['dlycaldt'].astype('datetime64[ns]')
fund_linked['rdq'] = fund_linked['rdq'].astype('datetime64[ns]')
# 必须先对两张表按 PERMNO 和 日期进行排序，这是 merge_asof 的前置要求
daily_stock = daily_stock.sort_values(['dlycaldt'])
fund_linked = fund_linked.sort_values(['rdq'])

# 使用 merge_asof 进行向后填充 (Forward-fill)
# 逻辑：对于每一天的日线数据，去寻找在这一天之前（刚好最近一次）公布的财报数据
master_data = pd.merge_asof(
    daily_stock,
    fund_linked,
    left_on='dlycaldt',
    right_on='rdq',
    by='permno',
    direction='backward'
)


# ==========================================
# 第五步：合并宏观数据 (Macro Features)
# ==========================================
print("5. 正在合并宏观外部特征...")
master_data = pd.merge(master_data, macro, left_on='dlycaldt', right_on='date', how='left')
master_data = master_data.drop(columns=['date'])
# 宏观数据周末如果有空缺，顺延填充
master_data[['10y_treasury', '3m_tbill', 'vix']] = master_data[['10y_treasury', '3m_tbill', 'vix']].ffill()


# 第六步：应用 S&P 500 历史成分股 Mask (消除幸存者偏差)
print("6. 正在应用 S&P 500 历史成分股过滤器...")

if 'mbrstartdt' in sp500.columns and 'mbrenddt' in sp500.columns:
    
    # 1. 瞬间挤干水分，只提取独一无二的纳入记录
    clean_sp500 = sp500[['permno', 'mbrstartdt', 'mbrenddt']].drop_duplicates().copy()
    
    # 2. 【关键修复：彻底消灭 1970 年穿越Bug】
    # 将数字强转为字符串，并切掉可能存在的 '.0' 小数点尾巴
    clean_sp500['mbrstartdt'] = clean_sp500['mbrstartdt'].astype(str).str.split('.').str[0]
    clean_sp500['mbrenddt'] = clean_sp500['mbrenddt'].astype(str).str.split('.').str[0]
    
    # 精准按照 年月日 (%Y%m%d) 解析！遇到 'nan' 则强制变为 NaT
    clean_sp500['mbrstartdt'] = pd.to_datetime(clean_sp500['mbrstartdt'], format='%Y%m%d', errors='coerce')
    clean_sp500['mbrenddt'] = pd.to_datetime(clean_sp500['mbrenddt'], format='%Y%m%d', errors='coerce').fillna(pd.to_datetime('today'))
    
    # 剔除那些连纳入日期都没有的彻底无效的废弃行
    clean_sp500 = clean_sp500.dropna(subset=['mbrstartdt'])
    
    master_permno = master_data['permno'].values
    master_date = master_data['dlycaldt'].values
    valid_mask = np.zeros(len(master_data), dtype=bool)
    
    # 提取纯净的 Numpy 数组
    sp_permnos = clean_sp500['permno'].values
    sp_starts = clean_sp500['mbrstartdt'].values
    sp_ends = clean_sp500['mbrenddt'].values
    
    total = len(sp_permnos)
    print(f"🔥 日期已修复！实际需要匹配的有效成分股记录：{total} 条")
    
    for count, (p, start, end) in enumerate(zip(sp_permnos, sp_starts, sp_ends), 1):
        mask = (master_permno == p) & (master_date >= start) & (master_date <= end)
        valid_mask |= mask  
        
        if count % 100 == 0 or count == total:
            print(f"处理进度: {count}/{total}", end='\r')

    print("\n✅ 过滤完成！")
    master_data = master_data[valid_mask].copy()

else:
    print("⚠️ 警告：缺少日期列，跳过历史剔除。")


# 完工清理
# 排序并重置索引
master_data = master_data.sort_values(['permno', 'dlycaldt']).reset_index(drop=True)

print(f"\n✅ 数据合并大功告成！最终大宽表包含 {len(master_data)} 行数据。")
print("前 5 行预览：")
print(master_data.head())


#master_data.to_csv('master_factor_dataset.csv', index=False)
#master_data.to_parquet('master_factor_dataset.parquet', index=False)
#master_data.to_pickle('master_factor_dataset.pkl')


开始构建多因子数据宽表...
1. 正在读取数据...
2. 正在处理 CCM 连结表...
3. 正在将财务数据翻译为 PERMNO...
4. 正在进行 Point-in-Time 频率合并 (这步可能需要几秒到十几秒)...
5. 正在合并宏观外部特征...
6. 正在应用 S&P 500 历史成分股过滤器 (去重秒杀版)...
🔥 水分已挤干！实际需要匹配的有效成分股记录仅剩：761 条！
处理进度: 761/761
✅ 过滤完成！

✅ 数据合并大功告成！最终大宽表包含 0 行数据。
前 5 行预览：
Empty DataFrame
Columns: [permno, hdrcusip, ticker, permco, dlycaldt, dlyprc, dlyret, dlyvol, dlylow, dlyhigh, shrout, costat, curcdq, datafmt, indfmt, consol, rdq, actq, ceqq, cheq, dlcq, lctq, niq, capxy, oancfy, 10y_treasury, 3m_tbill, vix]
Index: []

[0 rows x 28 columns]
